# Fig. 5 and S5: bridging unpaired RNA and ATAC cohorts

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashford-A/UniVI/blob/main/docs/reproducibility/api/fig5_bridge.ipynb)

A paired 10x Multiome reference is trained, then two independent single-modality PBMC cohorts are projected into it: scRNA-seq (Ding et al. 2020) through the RNA encoder and scATAC-seq (Satpathy et al. 2019) through the ATAC encoder. A lightweight classifier head is then refined on the reference labels and used to annotate the projected cells (Supplemental Fig. S5). Settings follow the archived notebook `UniVI_manuscript_GR-Figure__5__Multiome_bridge_mapping_and_fine-tuning.ipynb`; preprocessing uses UniVI's fitted preprocessors, so results will be close to, not identical with, the published ones.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "univi[tutorials]>=1.1"

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import torch
from sklearn.neighbors import KNeighborsClassifier

import univi.datasets as uds
from univi import (ClassHeadConfig, ModalityConfig, RefinementConfig, TrainingConfig, UniVIConfig,
                   UniVIMultiModalVAE, UniVIRefiner, UniVITrainer, predict_heads_adata)
from univi.evaluation import encode_adata, encode_fused_adata_pair
from univi.preprocessing import ATACPreprocessor, RNAPreprocessor
from univi.utils.seed import set_seed
from univi.workflows import make_loader, stack_embeddings

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
set_seed(0)

In [ ]:
N_EPOCHS = 5000        # archived notebook: 5000 with early stopping (patience 150)
REFINE_EPOCHS = 3000   # classifier refinement: 300 head-only warmup epochs, then encoders + head
N_HVG = 2000
N_LSI = 100
LABEL_KEY = "celltype_harmonized_coarse"

## Data

`rna`/`atac` are the paired reference; `ding_rna` and `satpathy_atac` are the query cohorts. Features are restricted to those the reference shares with each query (genes with the Ding cohort, peaks with the Satpathy cohort), so the fitted preprocessors can be applied to the queries unchanged. Highly variable genes are selected on reference training cells.

In [ ]:
data = uds.load("pbmc_multiome_bridge")
rna, atac, ding, sat = data["rna"], data["atac"], data["ding_rna"], data["satpathy_atac"]
genes = rna.var_names.intersection(ding.var_names)
peaks = atac.var_names.intersection(sat.var_names)
rna, atac = rna[:, genes].copy(), atac[:, peaks].copy()
print(f"shared genes {len(genes)}, shared peaks {len(peaks)}; reference {rna.n_obs}, Ding {ding.n_obs}, Satpathy {sat.n_obs}")

if "split" in rna.obs:
    splits = {k: np.flatnonzero(rna.obs["split"].to_numpy() == k) for k in ("train", "val")}
else:                                                    # archived notebook: random 90/10, seed 42
    idx = np.random.default_rng(42).permutation(rna.n_obs)
    splits = {"train": np.sort(idx[: int(0.9 * len(idx))]), "val": np.sort(idx[int(0.9 * len(idx)):])}

rna_prep = RNAPreprocessor(n_hvg=N_HVG, scale=True).fit(rna[splits["train"]])
atac_prep = ATACPreprocessor(n_components=N_LSI, drop_first=False, scale=True).fit(atac[splits["train"]])
ref = {"rna": rna_prep.transform(rna), "atac": atac_prep.transform(atac)}
train = {m: a[splits["train"]] for m, a in ref.items()}
val = {m: a[splits["val"]] for m, a in ref.items()}
ding_pp = rna_prep.transform(ding[:, genes])
sat_pp = atac_prep.transform(sat[:, peaks])

## Train the reference

In [ ]:
cfg = UniVIConfig(
    latent_dim=30, beta=1.0, gamma=5.0, encoder_dropout=0.25, decoder_dropout=0.05,
    kl_anneal_start=0, kl_anneal_end=25, align_anneal_start=15, align_anneal_end=40,
    modalities=[
        ModalityConfig("rna", train["rna"].n_vars, [512, 256, 128], [128, 256, 512], likelihood="gaussian"),
        ModalityConfig("atac", train["atac"].n_vars, [128, 64], [64, 128], likelihood="gaussian"),
    ],
)
model = UniVIMultiModalVAE(cfg, loss_mode="v1", v1_recon="avg", normalize_v1_terms=True)
UniVITrainer(
    model, make_loader(train, batch_size=256, shuffle=True, drop_last=True), make_loader(val, batch_size=1024),
    TrainingConfig(n_epochs=N_EPOCHS, batch_size=256, lr=1e-3, weight_decay=1e-4, device=device,
                   early_stopping=True, patience=150, best_epoch_warmup=40, log_every=100),
).fit();

## Project the unimodal cohorts

In [ ]:
joint = stack_embeddings(model, [("multiome", "rna", ref["rna"]), ("multiome", "atac", ref["atac"]),
                                 ("ding", "rna", ding_pp), ("satpathy", "atac", sat_pp)], device=device)
sc.pp.neighbors(joint, use_rep="X_univi", n_neighbors=30)
sc.tl.umap(joint, random_state=0)
color = ["block"] + ([LABEL_KEY] if LABEL_KEY in joint.obs else [])
sc.pl.umap(joint, color=color, wspace=0.45, legend_fontsize=7)

k-NN label transfer from the reference's fused embedding gives a first annotation of the query cells:

In [ ]:
z_ref = encode_fused_adata_pair(model, adata_by_mod=ref, device=device, write_to_adatas=False)["mu"]
knn = KNeighborsClassifier(n_neighbors=15).fit(z_ref, ref["rna"].obs[LABEL_KEY].astype(str))
queries = {"ding": (ding_pp, "rna"), "satpathy": (sat_pp, "atac")}
for name, (q, mod) in queries.items():
    q.obs["knn_label"] = knn.predict(encode_adata(model, q, modality=mod, device=device, latent="modality_mean"))
    if LABEL_KEY in q.obs:
        print(f"{name}: k-NN agreement with provided labels = {(q.obs['knn_label'] == q.obs[LABEL_KEY].astype(str)).mean():.3f}")

## Supervised refinement with a classifier head (Supplemental Fig. S5)

The head ([64, 64, 32], LayerNorm, dropout 0.1) is trained on reference labels, first alone and then together with the encoders at a small learning rate; decoders stay frozen. RNA-only and ATAC-only views of the reference teach it to classify from either modality.

In [ ]:
classes = sorted(ref["rna"].obs[LABEL_KEY].astype(str).unique())
code = {c: i for i, c in enumerate(classes)}
y = {k: {LABEL_KEY: a.obs[LABEL_KEY].astype(str).map(code).to_numpy()} for k, a in [("train", train["rna"]), ("val", val["rna"])]}
model.add_classification_head(ClassHeadConfig(LABEL_KEY, n_classes=len(classes), hidden_dims=[64, 64, 32],
                                              dropout=0.1, batchnorm=False, layernorm=True), label_names=classes)
refiner = UniVIRefiner(
    model,
    [make_loader({"rna": train["rna"]}, labels=y["train"], batch_size=256, shuffle=True),
     make_loader({"atac": train["atac"]}, labels=y["train"], batch_size=256, shuffle=True)],
    [make_loader({"rna": val["rna"]}, labels=y["val"], batch_size=1024),
     make_loader({"atac": val["atac"]}, labels=y["val"], batch_size=1024)],
    device=device,
    config=RefinementConfig(max_epochs=REFINE_EPOCHS, warmup_epochs=min(300, REFINE_EPOCHS),
                            lr_head=3e-4, lr_encoder=1e-5, log_every=100),
)
refiner.fit();

In [ ]:
for name, (q, mod) in queries.items():
    proba = predict_heads_adata(model, q, mod, device=device)[LABEL_KEY]
    q.obs["head_label"] = pd.Categorical(np.asarray(classes)[proba.argmax(1)])
    q.obs["head_confidence"] = proba.max(1)
    q.obsm["X_univi"] = encode_adata(model, q, modality=mod, device=device, latent="modality_mean")
    msg = f"{name}: head vs k-NN agreement {(q.obs['head_label'].astype(str) == q.obs['knn_label']).mean():.3f}"
    if LABEL_KEY in q.obs:
        msg += f"; head vs provided labels {(q.obs['head_label'].astype(str) == q.obs[LABEL_KEY].astype(str)).mean():.3f}"
    print(msg)

sc.pp.neighbors(ding_pp, use_rep="X_univi")
sc.tl.umap(ding_pp, random_state=0)
sc.pl.umap(ding_pp, color=["head_label", "head_confidence"], wspace=0.45, legend_fontsize=7)
markers = [g for g in ["MS4A1", "CD79A", "FCGR3A", "LST1", "IL7R", "TRAC", "CD8A", "NKG7", "S100A8", "LYZ", "CLEC10A"]
           if g in ding_pp.var_names]
sc.pl.umap(ding_pp, color=markers, ncols=4, vmax="p99", cmap="viridis")